# Lab 4.4 &mdash; Langfuse MCP in a ReAct Agent

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Connect to the Langfuse MCP server from Python and list what it offers
- Wrap the tools you want as LangChain tools
- Hand them to a ReAct agent and watch it reason, call and answer

> **How this lab works.** You write real LangChain and MCP code. Fill every `BLANK`, then run
> the **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a `@tool`, an argument schema, a `ToolMessage`, an `mcp.types.Tool`), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code
> in front of the sandbox model; that is the part worth watching. The score line is feedback,
> not a grade.

> **Short one.** Labs 4.1&ndash;4.3 gave MCP servers to an agent someone else wrote.
> This is the same thing in twenty lines of your own code.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and that reasoning is billed as completion
# tokens. It is off here because tool selection is a short decision and you will make a lot
# of them today. Pass think=True to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

## Concept

A ReAct agent loops: **reason &rarr; act &rarr; observe**, until it can answer. `act` means calling
a tool, and LangChain does not care where that tool came from.

So putting an MCP server inside a ReAct agent is one translation step:

```
MCP tools/list  ->  StructuredTool  ->  create_agent(tools=[...])
```

`name`, `description` and `inputSchema` come straight across &mdash; `StructuredTool` accepts JSON
Schema as `args_schema`, so there is no conversion to write.

## Step 1 &mdash; Talk to the server

Two calls: `initialize`, then `tools/list`. Everything the sandbox needs is already in your
environment.

In [ ]:
import base64, urllib.request

LF_URL  = os.environ.get("LANGFUSE_HOST", "").rstrip("/") + "/api/public/mcp"
LF_AUTH = base64.b64encode(f"{os.environ.get('LANGFUSE_PUBLIC_KEY','')}:"
                           f"{os.environ.get('LANGFUSE_SECRET_KEY','')}".encode()).decode()
_sid = {"v": None}


def langfuse_ready() -> bool:
    return all(os.environ.get(k) for k in
               ("LANGFUSE_HOST", "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"))


def mcp(method: str, params: dict = None) -> dict:
    """One JSON-RPC call to the MCP server. Returns the whole envelope."""
    body = json.dumps({"jsonrpc": "2.0", "id": 1, "method": method, "params": params or {}}).encode()
    req = urllib.request.Request(LF_URL, data=body, method="POST")
    req.add_header("Content-Type", "application/json")
    req.add_header("Accept", "application/json, text/event-stream")
    req.add_header("Authorization", "Basic " + LF_AUTH)
    if _sid["v"]:
        req.add_header("Mcp-Session-Id", _sid["v"])
    with urllib.request.urlopen(req, timeout=120) as r:
        raw, sid = r.read().decode(), r.headers.get("mcp-session-id")
    if sid:
        _sid["v"] = sid
    for line in raw.splitlines():
        if line.startswith("data:"):
            raw = line[5:].strip()
            break
    return json.loads(raw)


if langfuse_ready():
    mcp("initialize", {"protocolVersion": "2025-06-18", "capabilities": {},
                       "clientInfo": {"name": "lab-4-4", "version": "1.0"}})
    published = guard(lambda: mcp("tools/list", {})["result"]["tools"], [])
    print(f"the server offers {len(published)} tools, for example:")
    for t in published[:5]:
        print(f"  {t['name']:24} {t.get('description','')[:56]}")
else:
    published = []
    print("Langfuse is not configured here. Set LANGFUSE_HOST / _PUBLIC_KEY / _SECRET_KEY.")

## Step 2 &mdash; Wrap the ones you want

You are not binding all of them. Pick the few that answer the question you care about &mdash; every
tool you hand the agent is one more thing it can choose wrongly, and one more schema it carries on
every turn.

Fill in the blank: **which field of the MCP spec is the sentence the model reads when deciding
whether to call this tool?**

In [ ]:
from langchain_core.tools import StructuredTool

WANTED = ["getMetricsSchema",            # which measures queryMetrics accepts
          "queryMetrics",               # aggregate: how many, how slow, by name
          "getObservationFieldSchema",  # which field names listObservations accepts
          "listObservations",           # find individual ones
          "getObservation"]             # fetch one in full


def as_langchain_tool(spec: dict) -> StructuredTool:
    """One MCP tool definition -> one LangChain tool."""
    def call(**kwargs):
        env = mcp("tools/call", {"name": spec["name"], "arguments": kwargs})
        if "error" in env:                       # let the model see failures, or it will loop
            return "ERROR: " + str(env["error"].get("message", env["error"]))[:400]
        parts = env.get("result", {}).get("content", [])
        return "".join(p.get("text", "") for p in parts)[:1500]

    return StructuredTool.from_function(
        func=call,
        name=spec["name"],
        description=spec[BLANK][:900],           # name? inputSchema? or the prose?
        args_schema=spec.get("inputSchema") or {"type": "object", "properties": {}},
    )


tools = [as_langchain_tool(s) for s in published if s["name"] in WANTED]
print("bridged:", [t.name for t in tools])

In [ ]:
# ---- Self-check: the objects you built. No model, no network ----
SPEC = {"name": "listPrompts",
        "description": "List prompts in the current Langfuse project.",
        "inputSchema": {"type": "object",
                        "properties": {"limit": {"type": "integer"}}, "required": []}}

check("the wrapper produces a LangChain tool",
      lambda: isinstance(as_langchain_tool(SPEC), StructuredTool))
check("the name survives",
      lambda: as_langchain_tool(SPEC).name == "listPrompts")
check("the description is the prose the model reads",
      lambda: as_langchain_tool(SPEC).description.startswith("List prompts"),
      "the name says what it is called; only this says when to use it")
check("MCP's JSON Schema became the tool's arguments",
      lambda: set(as_langchain_tool(SPEC).args) == {"limit"})
score()

## Step 3 &mdash; Give them to a ReAct agent

`create_agent` builds the reason&ndash;act&ndash;observe loop. It has no idea these tools speak MCP.

In [ ]:
from langchain.agents import create_agent

SYSTEM = (
    "You answer questions about a Langfuse project using the tools provided.\n"
    "Call getMetricsSchema before any queryMetrics call, and getObservationFieldSchema "
    "before any listObservations call. Use only the names they list -- guessing a field "
    "or measure name costs several wasted turns.\n"
    "Latency is measured in milliseconds.\n"
    "If a tool returns an ERROR, read it and correct your arguments. Be brief."
)


def ask(question: str) -> str:
    agent = create_agent(model=get_llm(), tools=tools, system_prompt=SYSTEM)
    out = agent.invoke({"messages": [("user", question)]})
    for m in out["messages"]:                       # show the act/observe steps
        for tc in (getattr(m, "tool_calls", None) or []):
            print("  act:", tc["name"], json.dumps(tc["args"])[:88])
    return out["messages"][-1].content


QUESTION = ("Find the slowest observation in the last 7 days, fetch it in full, "
            "and explain in three sentences what it was doing.")

if llm_ready() and tools:
    print(guard(lambda: ask(QUESTION)))
else:
    print("Run it for real needs the model and Langfuse. See the setup cell.")

## What you just saw

Read the `act:` lines. That one English sentence became a chain: ask the schema what exists,
`queryMetrics` to rank observations by latency, `listObservations` to identify the slow one,
`getObservation` to read it in full &mdash; then the model wrote the summary. **You sequenced none
of that.** You handed it five tools and a question in English.

On this model and this question, expect **fifteen to twenty `act:` lines and two to three
minutes**, with visible retries: the model proposes an argument, the server returns `ERROR:`, it
reads it and tries again. That recovery is why the wrapper *returns* error text instead of raising
&mdash; a raise ends the turn, a returned string lets the model correct itself. The agent is
slower and more argumentative than the four lines of summary suggest. Watch the middle, not just
the answer.

`getObservationFieldSchema` is in `WANTED` for a reason worth knowing. An earlier version bound
only the other four, and the model tried to call `getObservationFieldSchema` anyway &mdash; a tool
it had never been given &mdash; then spent six turns guessing `start_time` versus `startTime`.
**It named the tool it was missing.** What an agent reaches for and cannot find is the best signal
you get about what to bind next.

And the honest result of binding it: the field-name guessing stopped, the answer got richer &mdash;
and the **turn count did not drop at all**. The model simply moved its uncertainty to the shape of
`queryMetrics`. Adding a tool removed one failure mode; it did not buy speed. Measure that before
you promise it to anyone.

That is the whole integration: **twenty lines, and an MCP server is just tools now.**

## Now interrogate it yourself

`ask()` is the entire interface. Edit `QUESTION` and re-run the cell, or call it directly:

```python
ask("Which observation names appear most often in the last hour?")
ask("Give me average and maximum latency by observation name for today.")
ask("List the three most recent observations with their name, type and latency.")
ask("Find the most recent GENERATION and tell me which model it used.")
ask("Are there any observations with level ERROR? Show me the most recent one in full.")
```

Questions answered by one aggregate finish in two or three steps. Ones that end in "&hellip;and
fetch it in full" chain three tools, because finding a thing and reading a thing are different
tools &mdash; the agent works that out from the descriptions, not from you.

## Your turn

- Add `listPrompts` to `WANTED` and ask something that needs it. One line, one more capability.
- Truncate the description to 30 characters and re-run. Same tools, worse choices &mdash; the prose
  was doing the work.
- Point the same wrapper at the Jira server from Lab 4.1. Nothing in it is Langfuse-specific.